# TalentMatch RAG — Experimentation & Performance Analysis

This notebook evaluates the resume matching pipeline across two dimensions:

1. **Retrieval accuracy** — Precision@K, Recall@K, and MRR on manually labelled job-description scenarios
2. **Latency** — End-to-end and per-stage timing for ingestion and query pipelines

**Prerequisites:** Run from the project root with the vector store already built (`python resume_rag.py --resume-dir resumes`).

Labelled ground truth lives in `data/eval_labels.json`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "eval_harness.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from eval_harness import (
    benchmark_latency,
    corpus_stats,
    evaluate_all_cases,
    load_job_description,
    load_labels,
    results_to_dataframe_rows,
)
from resume_rag import build_vector_store

print(f"Project root: {PROJECT_ROOT}")

## 1. Corpus Overview

Confirm ingestion state before running retrieval experiments.

In [ ]:
stats = corpus_stats()
if stats["total_chunks"] == 0:
    print("Vector store empty — building now...")
    build_vector_store(str(PROJECT_ROOT / "resumes"))
    stats = corpus_stats()

corpus_df = pd.DataFrame([stats])
display(corpus_df[["total_resumes", "indexed_resumes", "total_chunks"]])

fig, ax = plt.subplots(figsize=(5, 3))
formats = list(stats["format_counts"].keys())
counts = list(stats["format_counts"].values())
ax.bar(formats, counts, color=["#4F46E5", "#7C3AED", "#06B6D4"][: len(formats)])
ax.set_title("Resume formats in corpus")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 2. Retrieval Accuracy Experiments

We evaluate against labelled scenarios in `data/eval_labels.json`.

- **Positive label:** `relevant` or `partial` (lenient mode per `docs/eval.md`)
- **Precision@K:** share of top-K results that are positively labelled
- **Recall@K:** share of all known positives found in top-K
- **MRR:** reciprocal rank of the first positive result

Target from project docs: lenient **Precision@3 ≥ 0.67** and **Recall@5 = 1.0** on the canonical ML engineer JD.

In [ ]:
eval_results = evaluate_all_cases(top_k=10)
metrics_df = pd.DataFrame(results_to_dataframe_rows(eval_results))
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

metrics_df.plot.bar(x="jd", y="precision@3", ax=axes[0], legend=False, color="#4F46E5")
axes[0].axhline(0.67, color="red", linestyle="--", label="target 0.67")
axes[0].set_title("Precision@3")
axes[0].set_ylim(0, 1)

metrics_df.plot.bar(x="jd", y="recall@5", ax=axes[1], legend=False, color="#7C3AED")
axes[1].axhline(1.0, color="red", linestyle="--", label="target 1.0")
axes[1].set_title("Recall@5")
axes[1].set_ylim(0, 1)

metrics_df.plot.bar(x="jd", y="mrr", ax=axes[2], legend=False, color="#06B6D4")
axes[2].set_title("Mean Reciprocal Rank")
axes[2].set_ylim(0, 1)

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 3. Qualitative Analysis — Top Matches per JD

Inspect ranked candidates, expected labels, and matched skills.

In [ ]:
for case in eval_results:
    print(f"\n=== {case['title']} ({case['jd_path']}) ===")
    top_df = pd.DataFrame(case["top_matches"])
    display(top_df[["candidate_name", "match_score", "matched_skills", "expected_label", "resume_path"]])

## 4. Latency Benchmarks

Timed over multiple runs on the canonical ML engineer JD (`docs/sample_jd.txt`).

Retrieval timing excludes Groq LLM reasoning so we isolate vector-search performance.
Add `GROQ_API_KEY` to `.env` if you also want to benchmark full `match_jobs()` with reasoning.

In [ ]:
canonical_jd = load_job_description("docs/sample_jd.txt")
latency = benchmark_latency(canonical_jd, runs=5)

latency_rows = []
for stage, values in latency.items():
    if stage == "runs":
        continue
    latency_rows.append({"stage": stage, **values})

latency_df = pd.DataFrame(latency_rows)
latency_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(latency_df["stage"], latency_df["mean_ms"], color="#4F46E5")
ax.errorbar(
    latency_df["stage"],
    latency_df["mean_ms"],
    yerr=latency_df["p95_ms"] - latency_df["mean_ms"],
    fmt="none",
    ecolor="#1E1B4B",
    capsize=4,
)
ax.set_title("Query pipeline latency (mean ms, p95 error bars)")
ax.set_ylabel("Milliseconds")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Findings & Discussion

### Retrieval accuracy
- The **canonical ML engineer** scenario surfaces strong matches (e.g. ML-focused resumes) near the top; MRR reflects how quickly the first labelled positive appears.
- **Precision@3** can look lower when unlabelled but semantically similar resumes occupy top slots — expand `data/eval_labels.json` for tighter measurement.
- The **frontend JD** tests domain shift: React/TypeScript resumes should rank above ML specialists.
- Must-have skill filtering removes candidates missing hard requirements (e.g. experience years or required skills).

### Latency
- **JD embedding** is the fastest stage (single sentence-transformers call).
- **Semantic search + hybrid filtering** dominates query time and stays sub-100ms on this corpus (~200 chunks).
- **Groq LLM reasoning** (not timed here) adds network latency per candidate when using `match_jobs()` in the API.

### Recommendations
1. Re-run ingestion after uploading new resumes before evaluating.
2. Expand labelled scenarios as the corpus grows.
3. For production, cache JD embeddings and batch LLM reasoning requests.